In [ ]:
# !pip install -q tensorflow keras scikit-learn scipy matplotlib seaborn spotipy

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from scipy import stats as sp_stats

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Conv2D, MaxPooling2D, Flatten,
    Dropout, LSTM, Bidirectional, concatenate, Layer
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
print(f"TensorFlow : {tf.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")


In [ ]:
# CELL 1: MOUNT GOOGLE DRIVE & SET PATHS

from google.colab import drive
drive.mount('/content/drive')

MUSE_DIR = "/content/drive/MyDrive/MUSE/Muse"

# The code will look for these files (in order of preference):
#   1. muse_enriched_spotify.csv  (if you enriched with Spotify features — best)
#   2. muse_v3.csv                (base MuSe download from Kaggle)
#   3. Any .csv file in the folder

RESULTS_DIR = "/content/drive/MyDrive/HABC_Results_MUSE"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Find the CSV file
csv_file = None
for preferred in ['muse_enriched_spotify.csv', 'muse_v3.csv']:
    candidate = os.path.join(MUSE_DIR, preferred)
    if os.path.exists(candidate):
        csv_file = candidate
        break

if csv_file is None:
    csvs = [f for f in os.listdir(MUSE_DIR) if f.endswith('.csv')]
    if csvs:
        csv_file = os.path.join(MUSE_DIR, csvs[0])
    else:
        raise FileNotFoundError(f"No CSV files found in {MUSE_DIR}")

print(f"Using dataset: {csv_file}")
print(f"Results dir:   {RESULTS_DIR}")



In [ ]:
# CELL 2: ATTENTION LAYER + ALL MODEL BUILDERS
###############################################################################

class AttentionLayer(Layer):
    """Temporal attention: learns which time steps matter most."""
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1), initializer="normal")
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1), initializer="zeros")
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        et = K.squeeze(K.tanh(K.dot(x, self.W) + self.b), axis=-1)
        at = K.softmax(et)
        at = K.expand_dims(at, axis=-1)
        output = x * at
        return K.sum(output, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        return super(AttentionLayer, self).get_config()


def build_habc_model(num_classes, time_steps, feature_dim, lr=0.001):
    """FULL HABC: CNN + BiLSTM + Attention."""
    cnn_inp = Input(shape=(1, time_steps, feature_dim), name='cnn_input')
    c = Conv2D(6, (3, 3), padding='same', activation='relu')(cnn_inp)
    c = MaxPooling2D(pool_size=(1, 2), strides=(1, 2), padding='same')(c)
    c = Flatten()(c)

    lstm_inp = Input(shape=(time_steps, feature_dim), name='lstm_input')
    l = Bidirectional(LSTM(64, return_sequences=True))(lstm_inp)
    l = AttentionLayer()(l)

    merged = concatenate([c, l])
    x = Dense(64, activation='relu')(merged)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=[cnn_inp, lstm_inp], outputs=out)
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  metrics=['accuracy'])
    return model


def build_cnn_only(num_classes, time_steps, feature_dim, lr=0.001):
    inp = Input(shape=(1, time_steps, feature_dim))
    x = Conv2D(6, (3,3), padding='same', activation='relu')(inp)
    x = MaxPooling2D(pool_size=(1,2), strides=(1,2), padding='same')(x)
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr), metrics=['accuracy'])
    return m


def build_bilstm_only(num_classes, time_steps, feature_dim, lr=0.001):
    inp = Input(shape=(time_steps, feature_dim))
    x = Bidirectional(LSTM(64, return_sequences=False))(inp)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr), metrics=['accuracy'])
    return m


def build_bilstm_attention_only(num_classes, time_steps, feature_dim, lr=0.001):
    inp = Input(shape=(time_steps, feature_dim))
    x = Bidirectional(LSTM(64, return_sequences=True))(inp)
    x = AttentionLayer()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr), metrics=['accuracy'])
    return m


def build_cnn_bilstm_no_attention(num_classes, time_steps, feature_dim, lr=0.001):
    cnn_inp = Input(shape=(1, time_steps, feature_dim))
    c = Conv2D(6, (3,3), padding='same', activation='relu')(cnn_inp)
    c = MaxPooling2D(pool_size=(1,2), strides=(1,2), padding='same')(c)
    c = Flatten()(c)
    lstm_inp = Input(shape=(time_steps, feature_dim))
    l = Bidirectional(LSTM(64, return_sequences=False))(lstm_inp)
    merged = concatenate([c, l])
    x = Dense(64, activation='relu')(merged)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    m = Model(inputs=[cnn_inp, lstm_inp], outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr), metrics=['accuracy'])
    return m


In [ ]:
# =============================================================================
# CELL 3: LOAD & PREPROCESS MuSe DATA (FIXED FOR COLUMN NAMES)
# =============================================================================

import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

def load_and_preprocess_muse(csv_path, max_samples=20000):
    print("  Reading CSV …")
    df = pd.read_csv(csv_path)
    print(f"  Raw CSV shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")

    # ─── Identify available feature columns (FIXED) ───
    # Dynamically find the correct V-A-D column names in your specific CSV
    muse_core = []
    for base in ['valence', 'arousal', 'dominance']:
        for variant in [base, f"{base}_tags", f"{base}_muse"]:
            if variant in df.columns:
                muse_core.append(variant)
                break

    # Spotify enrichment columns (present if enriched)
    spotify_cols = [
        'danceability', 'energy', 'acousticness', 'instrumentalness',
        'liveness', 'speechiness', 'tempo', 'loudness',
        'valence_spotify', 'mode', 'key'
    ]

    # Handle column naming conflicts
    if 'valence' in df.columns and 'valence_spotify' not in df.columns:
        for alt in ['sp_valence', 'spotify_valence']:
            if alt in df.columns:
                df.rename(columns={alt: 'valence_spotify'}, inplace=True)
                break

    if 'danceability' in df.columns and 'valence' in df.columns:
        if 'valence_spotify' not in df.columns:
            if df['valence'].max() <= 1.0:
                df.rename(columns={'valence': 'valence_spotify'}, inplace=True)
                print("  NOTE: valence column appears to be Spotify's. "
                      "Using arousal/dominance for quadrant computation.")

    available_features = []
    for col in muse_core + spotify_cols:
        if col in df.columns:
            available_features.append(col)

    # If very few features, also grab any other numeric columns
    if len(available_features) < 5:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        skip_cols = {'lastfm_url', 'spotify_id', 'musicbrainz_id', 'number_of_tags'}
        for col in numeric_cols:
            if col not in available_features and col not in skip_cols:
                available_features.append(col)

    print(f"\n  Using {len(available_features)} features: {available_features}")

    # ─── Filter rows with valid data ───
    df_clean = df[available_features].dropna()
    print(f"  After dropping NaN: {len(df_clean)} rows")

    # ─── Create emotion quadrant labels (FIXED) ───
    val_col = None
    aro_col = None

    for candidate in ['valence', 'valence_tags', 'valence_muse']:
        if candidate in df.columns:
            val_col = candidate
            break

    for candidate in ['arousal', 'arousal_tags', 'arousal_muse']:
        if candidate in df.columns:
            aro_col = candidate
            break

    if val_col is None or aro_col is None:
        raise ValueError(f"Cannot find valence/arousal columns. "
                         f"Available: {list(df.columns)}")

    print(f"  Valence column: '{val_col}' (range: {df[val_col].min():.2f} – {df[val_col].max():.2f})")
    print(f"  Arousal column: '{aro_col}' (range: {df[aro_col].min():.2f} – {df[aro_col].max():.2f})")

    val_median = df_clean[val_col].median() if val_col in df_clean.columns else df[val_col].median()
    aro_median = df_clean[aro_col].median() if aro_col in df_clean.columns else df[aro_col].median()

    val_values = df.loc[df_clean.index, val_col].values
    aro_values = df.loc[df_clean.index, aro_col].values

    labels = np.zeros(len(df_clean), dtype=int)
    labels[(val_values >= val_median) & (aro_values >= aro_median)] = 0  # Happy
    labels[(val_values <  val_median) & (aro_values >= aro_median)] = 1  # Angry
    labels[(val_values <  val_median) & (aro_values <  aro_median)] = 2  # Sad
    labels[(val_values >= val_median) & (aro_values <  aro_median)] = 3  # Relaxed

    print(f"\n  Emotion quadrant distribution:")
    for i, name in enumerate(['Happy', 'Angry', 'Sad', 'Relaxed']):
        print(f"    {name} (Q{i+1}): {np.sum(labels == i)} samples")

    X_raw = df_clean[available_features].values.astype('float32')

    if len(X_raw) > max_samples:
        print(f"\n  Subsampling from {len(X_raw)} to {max_samples} (balanced) …")
        from sklearn.utils import resample
        indices = []
        samples_per_class = max_samples // 4
        for cls in range(4):
            cls_idx = np.where(labels == cls)[0]
            n_take = min(samples_per_class, len(cls_idx))
            chosen = resample(cls_idx, n_samples=n_take, replace=False, random_state=42)
            indices.extend(chosen)
        indices = np.array(sorted(indices))
        X_raw = X_raw[indices]
        labels = labels[indices]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw).astype('float32')

    n_features = X_scaled.shape[1]
    TIME_STEPS = 10   # artificial temporal dimension

    X_seq = np.tile(X_scaled[:, np.newaxis, :], (1, TIME_STEPS, 1))

    feature_groups = {}
    feat_to_idx = {name: i for i, name in enumerate(available_features)}

    # Group A: Sentiment (V-A-D from MuSe) (FIXED)
    group_a = [feat_to_idx[f] for f in muse_core if f in feat_to_idx]
    if group_a:
        feature_groups['Sentiment (V-A-D)'] = group_a

    group_b = [feat_to_idx[f] for f in ['energy', 'loudness', 'tempo'] if f in feat_to_idx]
    if group_b:
        feature_groups['Energy (energy, loudness, tempo)'] = group_b

    group_c = [feat_to_idx[f] for f in ['danceability', 'speechiness', 'liveness'] if f in feat_to_idx]
    if group_c:
        feature_groups['Rhythm (dance, speech, live)'] = group_c

    group_d = [feat_to_idx[f] for f in ['acousticness', 'instrumentalness', 'mode', 'key'] if f in feat_to_idx]
    if group_d:
        feature_groups['Tonal (acoustic, instr, mode, key)'] = group_d

    feature_groups['All features'] = list(range(n_features))

    print(f"\n  Feature groups for ablation:")
    for gname, gidx in feature_groups.items():
        fnames = [available_features[i] for i in gidx]
        print(f"    {gname}: {fnames}")

    return X_seq, labels, available_features, feature_groups, TIME_STEPS


# ──────────── Load the data ────────────

print("=" * 60)
print("Loading MuSe dataset …")
print("=" * 60)

# Make sure 'csv_file' is defined above in your notebook (e.g., csv_file = r"C:\Prashant\muse\muse_v3.csv")
raw_X, raw_y, FEATURE_NAMES, FEATURE_GROUPS, TIME_STEPS = load_and_preprocess_muse(
    csv_file, max_samples=20000
)

FEATURE_DIM = raw_X.shape[2]
NUM_CLASSES = len(np.unique(raw_y))

CLASS_NAMES = {0: 'Happy', 1: 'Angry', 2: 'Sad', 3: 'Relaxed'}
CLASS_LIST = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]

print(f"\n{'='*60}")
print(f"  MuSe Data Summary")
print(f"{'='*60}")
print(f"  Total samples   : {len(raw_X)}")
print(f"  Time steps      : {TIME_STEPS}")
print(f"  Features/step    : {FEATURE_DIM}")
print(f"  Classes          : {NUM_CLASSES} → {CLASS_NAMES}")
print(f"  Class counts     : {dict(Counter(raw_y))}")
print(f"  X shape          : {raw_X.shape}")

# Prepare inputs
X_cnn_all = raw_X.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
X_lstm_all = raw_X.copy()
y_encoded = to_categorical(raw_y, num_classes=NUM_CLASSES)

print(f"\n  CNN  input : {X_cnn_all.shape}")
print(f"  LSTM input : {X_lstm_all.shape}")
print(f"  Labels     : {y_encoded.shape}")

# ─── Train / Test split (stratified, 70/30) ───
(X_train, X_test, y_train_raw, y_test_raw) = train_test_split(
    raw_X, raw_y, test_size=0.3, random_state=42, stratify=raw_y
)

X_cnn_train  = X_train.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
X_lstm_train = X_train.copy()
X_cnn_test   = X_test.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
X_lstm_test  = X_test.copy()

y_train = to_categorical(y_train_raw, num_classes=NUM_CLASSES)
y_test  = to_categorical(y_test_raw, num_classes=NUM_CLASSES)

print(f"\n  Train: {X_train.shape[0]} samples")
print(f"  Test:  {X_test.shape[0]} samples")
print(f"  Train class dist: {dict(Counter(y_train_raw))}")
print(f"  Test  class dist: {dict(Counter(y_test_raw))}")

In [ ]:
# CELL 4: SANITY CHECK — Train HABC on MuSe (optional)
###############################################################################

print("\n" + "=" * 60)
print("SANITY CHECK: Training HABC on MuSe data")
print("=" * 60)

K.clear_session()
model_sanity = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
model_sanity.summary()

es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
history = model_sanity.fit(
    [X_cnn_train, X_lstm_train], y_train,
    epochs=100, batch_size=64, callbacks=[es], verbose=1
)

# Evaluate on test set
score = model_sanity.evaluate([X_cnn_test, X_lstm_test], y_test, verbose=0)
print(f"\nTest accuracy (sanity): {score[1]*100:.2f}%")

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], linewidth=2)
ax1.set(title='Training Loss', xlabel='Epoch', ylabel='Loss')
ax1.grid(True, alpha=0.3)
ax2.plot(history.history['accuracy'], linewidth=2, color='green')
ax2.set(title='Training Accuracy', xlabel='Epoch', ylabel='Accuracy')
ax2.grid(True, alpha=0.3)
plt.suptitle('HABC Training on MuSe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/sanity_training_curves.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# CELL 5: EXPERIMENT 9 — FEATURE ABLATION
###############################################################################

print("\n" + "=" * 60)
print("EXPERIMENT 9: Feature Ablation — MuSe")
print("=" * 60)
print("Testing which feature groups contribute most to emotion recognition.\n")

ablation_results = {}

for group_name, col_indices in FEATURE_GROUPS.items():
    print(f"  Training with: {group_name} ({len(col_indices)} features) …")

    # Extract only this feature group
    X_train_sub = X_train[:, :, col_indices]  # (N, TIME_STEPS, subset_dim)
    X_test_sub  = X_test[:, :, col_indices]
    sub_dim = len(col_indices)

    X_cnn_tr = X_train_sub.reshape(-1, 1, TIME_STEPS, sub_dim)
    X_lstm_tr = X_train_sub.copy()
    X_cnn_te = X_test_sub.reshape(-1, 1, TIME_STEPS, sub_dim)
    X_lstm_te = X_test_sub.copy()

    K.clear_session()
    model_sub = build_habc_model(NUM_CLASSES, TIME_STEPS, sub_dim)
    es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
    model_sub.fit([X_cnn_tr, X_lstm_tr], y_train,
                  epochs=100, batch_size=64, verbose=0, callbacks=[es])

    y_pred = np.argmax(model_sub.predict([X_cnn_te, X_lstm_te], verbose=0), axis=1)

    acc  = accuracy_score(y_test_raw, y_pred)
    mf1  = f1_score(y_test_raw, y_pred, average='macro', zero_division=0)
    prec = precision_score(y_test_raw, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test_raw, y_pred, average='weighted', zero_division=0)

    feature_list = [FEATURE_NAMES[i] for i in col_indices]
    ablation_results[group_name] = {
        'accuracy': float(acc),
        'macro_f1': float(mf1),
        'precision': float(prec),
        'recall': float(rec),
        'n_features': len(col_indices),
        'features': feature_list
    }
    print(f"    → Acc={acc*100:.2f}%  Macro-F1={mf1*100:.2f}%  "
          f"Prec={prec*100:.2f}%  Rec={rec*100:.2f}%\n")

# ── Print results table ──
print(f"\n{'='*85}")
print(f"  {'Feature Group':<40} {'#Feat':>6} {'Accuracy':>10} {'Macro-F1':>10} {'Precision':>10}")
print(f"{'='*85}")
for gname, r in ablation_results.items():
    tag = " ◀" if "All" in gname else ""
    print(f"  {gname:<40} {r['n_features']:>5}  {r['accuracy']*100:>9.2f}% "
          f"{r['macro_f1']*100:>9.2f}% {r['precision']*100:>9.2f}%{tag}")
print(f"{'='*85}")

# ── Bar chart ──
fig, ax = plt.subplots(figsize=(10, 6))
names = list(ablation_results.keys())
accs = [ablation_results[n]['accuracy']*100 for n in names]
f1s  = [ablation_results[n]['macro_f1']*100 for n in names]
x = np.arange(len(names))
w = 0.35

b1 = ax.bar(x - w/2, accs, w, label='Accuracy (%)', color='#2196F3')
b2 = ax.bar(x + w/2, f1s,  w, label='Macro-F1 (%)',  color='#FF9800')

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Feature Ablation — HABC on MuSe', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)

for b in b1:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
            f'{b.get_height():.1f}', ha='center', fontsize=8)
for b in b2:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
            f'{b.get_height():.1f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp9_feature_ablation.png", dpi=300, bbox_inches='tight')
plt.show()

with open(f"{RESULTS_DIR}/exp9_feature_ablation.json", "w") as f:
    json.dump(ablation_results, f, indent=2)
print(f"Saved → exp9_feature_ablation.json + exp9_feature_ablation.png")

# ── Feature importance summary ──
print("\n  INSIGHT FOR PAPER:")
sorted_groups = sorted(
    [(n, r['accuracy']) for n, r in ablation_results.items() if 'All' not in n],
    key=lambda x: x[1], reverse=True
)
for rank, (gname, acc) in enumerate(sorted_groups, 1):
    print(f"  #{rank} {gname}: {acc*100:.2f}%")
print(f"  Combined (All features): {ablation_results.get('All features', {}).get('accuracy', 0)*100:.2f}%")


In [ ]:
#  CELL 6: EXPERIMENT 8 — CONFUSION MATRIX
###############################################################################

print("\n" + "=" * 60)
print("EXPERIMENT 8: Confusion Matrix — MuSe")
print("=" * 60)

# Train HABC on full features
K.clear_session()
model_cm = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
model_cm.fit([X_cnn_train, X_lstm_train], y_train,
             epochs=100, batch_size=64, verbose=0, callbacks=[es])

y_pred_cm = np.argmax(model_cm.predict([X_cnn_test, X_lstm_test], verbose=0), axis=1)
y_true_cm = y_test_raw

# ── Raw confusion matrix ──
cm = confusion_matrix(y_true_cm, y_pred_cm)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=CLASS_LIST, yticklabels=CLASS_LIST,
            linewidths=0.5, linecolor='lightgray')
ax.set(xlabel='Predicted', ylabel='True',
       title='Confusion Matrix — HABC on MuSe')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp8_confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

# ── Normalized confusion matrix ──
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues', ax=ax,
            xticklabels=CLASS_LIST, yticklabels=CLASS_LIST,
            linewidths=0.5, linecolor='lightgray', vmin=0, vmax=1)
ax.set(xlabel='Predicted', ylabel='True',
       title='Normalized Confusion Matrix — HABC on MuSe')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp8_confusion_matrix_norm.png", dpi=300, bbox_inches='tight')
plt.show()

overall_acc = accuracy_score(y_true_cm, y_pred_cm)
print(f"\n  Overall test accuracy: {overall_acc*100:.2f}%")
print(f"  Saved → exp8_confusion_matrix.png + exp8_confusion_matrix_norm.png")


In [ ]:
# CELL 7: EXPERIMENT 10 — CLASS-WISE METRICS
###############################################################################

print("\n" + "=" * 60)
print("EXPERIMENT 10: Class-Wise Precision, Recall, F1 — MuSe")
print("=" * 60)

report = classification_report(y_true_cm, y_pred_cm,
                               target_names=CLASS_LIST,
                               output_dict=True, zero_division=0)

# ── Print table ──
print(f"\n{'='*65}")
print(f"  {'Emotion':<15} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print(f"{'='*65}")

for cls_name in CLASS_LIST:
    r = report[cls_name]
    print(f"  {cls_name:<15} {r['precision']*100:>9.2f}% {r['recall']*100:>9.2f}% "
          f"{r['f1-score']*100:>9.2f}% {r['support']:>9.0f}")

print(f"  {'-'*60}")
for avg in ['macro avg', 'weighted avg']:
    r = report[avg]
    print(f"  {avg.title():<15} {r['precision']*100:>9.2f}% {r['recall']*100:>9.2f}% "
          f"{r['f1-score']*100:>9.2f}% {r['support']:>9.0f}")
print(f"{'='*65}")
print(f"\n  Overall Accuracy: {overall_acc*100:.2f}%")

# Full report
print(f"\n--- Full Classification Report ---")
print(classification_report(y_true_cm, y_pred_cm,
                            target_names=CLASS_LIST, digits=4, zero_division=0))

# ── Per-class F1 bar chart ──
fig, ax = plt.subplots(figsize=(8, 5))
f1_vals = [report[c]['f1-score']*100 for c in CLASS_LIST]
colors = ['#4CAF50' if v >= 60 else '#FF9800' if v >= 40 else '#F44336' for v in f1_vals]

bars = ax.bar(CLASS_LIST, f1_vals, color=colors, edgecolor='gray', linewidth=0.5)
ax.axhline(np.mean(f1_vals), color='blue', ls='--', alpha=0.7,
           label=f'Mean F1 = {np.mean(f1_vals):.1f}%')
ax.set(ylabel='F1-Score (%)', title='Per-Class F1-Score — HABC on MuSe')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp10_per_class_f1.png", dpi=300, bbox_inches='tight')
plt.show()

# ── Precision-Recall-F1 grouped bar chart ──
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(CLASS_LIST))
w = 0.25

precs = [report[c]['precision']*100 for c in CLASS_LIST]
recs  = [report[c]['recall']*100    for c in CLASS_LIST]

ax.bar(x - w, precs, w, label='Precision', color='#2196F3')
ax.bar(x,     recs,  w, label='Recall',    color='#4CAF50')
ax.bar(x + w, f1_vals, w, label='F1-Score', color='#FF9800')

ax.set_ylabel('Score (%)')
ax.set_title('Per-Class Metrics — HABC on MuSe', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_LIST)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp10_per_class_prec_rec_f1.png", dpi=300, bbox_inches='tight')
plt.show()

exp10 = {
    "experiment": "Exp10_ClassWise", "dataset": "MuSe",
    "overall_accuracy": float(overall_acc),
    "report": report
}
with open(f"{RESULTS_DIR}/exp10_classwise.json", "w") as f:
    json.dump(exp10, f, indent=2)
print(f"Saved → exp10_classwise.json + exp10_per_class_f1.png + exp10_per_class_prec_rec_f1.png")



In [ ]:
# CELL 8: FINAL SUMMARY
###############################################################################

print("\n" + "=" * 70)
print("  FINAL SUMMARY — ALL MuSe EXPERIMENTS COMPLETE")
print("=" * 70)

print(f"""
┌───────────────────────────────────────────────────────────────────────┐
│  Dataset: MuSe │ Samples: {len(raw_X):,} │ Classes: {NUM_CLASSES} │ Features: {FEATURE_DIM}       │
│  Emotion quadrants: Happy, Angry, Sad, Relaxed                       │
├───────────────────────────────────────────────────────────────────────┤
│  Exp 9: Feature Ablation                                             │""")

for gname, r in ablation_results.items():
    tag = " ◀ full" if "All" in gname else ""
    print(f"│    {gname:<38} Acc={r['accuracy']*100:5.2f}% F1={r['macro_f1']*100:5.2f}%{tag:<6}│")

print(f"""├───────────────────────────────────────────────────────────────────────┤
│  Exp 8:  Confusion matrices saved ✓                                  │
│  Exp 10: Class-wise metrics saved ✓                                  │
│          Overall accuracy: {overall_acc*100:.2f}%                                     │
├───────────────────────────────────────────────────────────────────────┤
│  All files saved to: {RESULTS_DIR}/               │
└───────────────────────────────────────────────────────────────────────┘
""")

# Master summary
master = {
    "model": "HABC (CNN + BiLSTM + Attention)",
    "dataset": "MuSe (Musical Sentiment)",
    "num_samples": int(len(raw_X)),
    "num_classes": NUM_CLASSES,
    "num_features": FEATURE_DIM,
    "feature_names": FEATURE_NAMES,
    "emotion_classes": CLASS_NAMES,
    "experiments": [
        "Exp9: Feature Ablation (per feature group)",
        "Exp8: Confusion Matrix (raw + normalized)",
        "Exp10: Class-Wise Metrics + charts",
    ],
    "output_files": [
        "sanity_training_curves.png",
        "exp9_feature_ablation.png",
        "exp8_confusion_matrix.png",
        "exp8_confusion_matrix_norm.png",
        "exp10_per_class_f1.png",
        "exp10_per_class_prec_rec_f1.png",
        "exp9_feature_ablation.json",
        "exp10_classwise.json",
        "master_summary.json",
    ]
}
with open(f"{RESULTS_DIR}/master_summary.json", "w") as f:
    json.dump(master, f, indent=2)

print("ALL MuSe EXPERIMENTS COMPLETE!")
print(f"Check Google Drive: {RESULTS_DIR}/")

